# Debugger da transferencia de reflexoes

Este notebook investiga ARC e GSM8K em 30 itens de teste por dataset, usando `phi4-mini` e `llama3-8b` (o modelo referido como ollama3.1:8b no pedido). As reflexoes do professor `gpt-5-petrobras` sao carregadas de `results/reflections`.

Resultados persistidos sao reportados como observados. Variacoes de `k`, limiar de similaridade e prompt so recebem acuracia depois que a rotina opcional de inferencia for executada.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]='5'
os.environ["RMCQ_DEBUGGER_BACKEND"] = "vllm"

In [ ]:
from pathlib import Path
import ast
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATASETS = ('arc', 'gsm8k')
STUDENTS = ('phi4-mini', 'llama3-8b','mistral-7b')
TEACHER = 'gpt-5-petrobras'
DEPTHS = ('simple', 'complex')
N_ITEMS = 50
K_VALUES = (1,2,3, 5)
THRESHOLDS = (None,0.50,0.60, 0.70, 0.80, 0.90)
print('ROOT =', ROOT)

ROOT = /home/rodrigo.flexa/Reflection-MCQ


## 1. Auditoria rapida do codigo e dos dados

A auditoria compila os modulos do `rmcq`, verifica o invariante de rotulos e confere se baseline, reflexoes e avaliacoes usam os mesmos UIDs. Ela tambem explicita lacunas da grade, em vez de trata-las como acuracia zero.

In [ ]:
def read_jsonl(path):
    if not path.exists():
        return []
    with path.open(encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]

audit = []
for path in sorted((ROOT / 'rmcq').rglob('*.py')):
    try:
        ast.parse(path.read_text(encoding='utf-8'))
        audit.append({'arquivo': str(path.relative_to(ROOT)), 'status': 'OK'})
    except SyntaxError as exc:
        audit.append({'arquivo': str(path.relative_to(ROOT)), 'status': f'ERRO: {exc}'})

schema_checks = []
for dataset in DATASETS:
    for split in ('train', 'test'):
        rows = read_jsonl(ROOT / 'data' / 'splits' / dataset / f'{split}.jsonl')
        bad = []
        for row in rows:
            labels = [c.get('label') for c in row.get('choices', [])]
            expected = [chr(65 + i) for i in range(len(labels))]
            if labels != expected or row.get('answerKey') not in labels:
                bad.append(row.get('uid'))
        schema_checks.append({'dataset': dataset, 'split': split, 'n': len(rows), 'rotulos_invalidos': len(bad), 'exemplos': bad[:3]})

display(pd.DataFrame(audit))
display(pd.DataFrame(schema_checks))
print('Lacuna esperada: os resultados persistidos desta rodada so contem k=3; k=1 e k=5 serao gerados pela rotina opcional.')

,arquivo,status
0,rmcq/__init__.py,OK
1,rmcq/__main__.py,OK
2,rmcq/backends/__init__.py,OK
3,rmcq/backends/azure.py,OK
4,rmcq/backends/base.py,OK
5,rmcq/backends/hf.py,OK
6,rmcq/backends/stub.py,OK
7,rmcq/backends/vllm_backend.py,OK
8,rmcq/cli.py,OK
9,rmcq/common.py,OK


,dataset,split,n,rotulos_invalidos,exemplos
0,arc,train,286,0,[]
1,arc,test,1171,0,[]
2,gsm8k,train,366,0,[]
3,gsm8k,test,1319,0,[]


Lacuna esperada: os resultados persistidos desta rodada so contem k=3; k=1 e k=5 serao gerados pela rotina opcional.


In [ ]:
def load_map(path):
    return {row['uid']: row for row in read_jsonl(path)}

def first_items(dataset, n=N_ITEMS):
    items = load_map(ROOT / 'data' / 'splits' / dataset / 'test.jsonl') 
    return list(items.values())[:n]

inventory = []
for student in STUDENTS:
    for dataset in DATASETS:
        base_path = ROOT / 'results' / 'baseline' / student / f'{dataset}_test.jsonl'
        base = load_map(base_path)
        selected = {item['uid'] for item in first_items(dataset)}
        for depth in DEPTHS:
            refl_path = ROOT / 'results' / 'reflections' / f'{student}__{TEACHER}__{depth}' / f'{dataset}.jsonl'
            eval_path = ROOT / 'results' / 'eval' / f'{student}__{TEACHER}__{depth}__k3' / f'{dataset}.jsonl'
            refl = load_map(refl_path)
            ev = load_map(eval_path)
            inventory.append({
                'student': student, 'dataset': dataset, 'depth': depth,
                'baseline_30': len(selected & base.keys()),
                'reflections': len(refl), 'eval_k3': len(selected & ev.keys()),
                'reflexoes_arquivo': refl_path.exists(), 'eval_arquivo': eval_path.exists(),
            })
display(pd.DataFrame(inventory))

,student,dataset,depth,baseline_30,reflections,eval_k3,reflexoes_arquivo,eval_arquivo
0,phi4-mini,arc,simple,50,286,50,True,True
1,phi4-mini,arc,complex,50,286,50,True,True
2,phi4-mini,gsm8k,simple,50,366,50,True,True
3,phi4-mini,gsm8k,complex,50,366,50,True,True
4,llama3-8b,arc,simple,50,286,50,True,True
5,llama3-8b,arc,complex,50,286,50,True,True
6,llama3-8b,gsm8k,simple,50,366,50,True,True
7,llama3-8b,gsm8k,complex,50,366,50,True,True
8,mistral-7b,arc,simple,50,286,50,True,True
9,mistral-7b,arc,complex,50,286,50,True,True


## 2. Tabela observada: baseline contra k=3

`SUCESSO` significa acuracia com reflexao estritamente maior que o baseline no mesmo subconjunto de 30 itens. Empate e `FRACASSO`, conforme o criterio pedido.

In [ ]:
def acc(rows):
    return sum(bool(r.get('is_correct')) for r in rows) / len(rows) if rows else np.nan

observed = []
for student in STUDENTS:
    for dataset in DATASETS:
        selected = [item['uid'] for item in first_items(dataset)]
        base = load_map(ROOT / 'results' / 'baseline' / student / f'{dataset}_test.jsonl')
        base_rows = [base[uid] for uid in selected if uid in base]
        base_acc = acc(base_rows)
        for depth in DEPTHS:
            ev = load_map(ROOT / 'results' / 'eval' / f'{student}__{TEACHER}__{depth}__k3' / f'{dataset}.jsonl')
            ev_rows = [ev[uid] for uid in selected if uid in ev]
            eval_acc = acc(ev_rows)
            sim = [((r.get('extra') or {}).get('top1_similarity')) for r in ev_rows]
            sim = [x for x in sim if x is not None]
            observed.append({
                'dataset': dataset, 'student': student, 'teacher': TEACHER,
                'prompt_variant': 'frozen_observed', 'depth': depth, 'k': 3,
                'similarity_threshold': None, 'n': len(ev_rows),
                'baseline_accuracy': base_acc, 'reflection_accuracy': eval_acc,
                'delta': eval_acc - base_acc if ev_rows else np.nan,
                'status': 'SUCESSO' if ev_rows and eval_acc > base_acc else 'FRACASSO',
                'mean_top1_similarity': np.mean(sim) if sim else np.nan,
            })
observed_df = pd.DataFrame(observed)
display(observed_df.style.format({'baseline_accuracy': '{:.3f}', 'reflection_accuracy': '{:.3f}', 'delta': '{:+.3f}', 'mean_top1_similarity': '{:.3f}'}))

,dataset,student,teacher,prompt_variant,depth,k,similarity_threshold,n,baseline_accuracy,reflection_accuracy,delta,status,mean_top1_similarity
0,arc,phi4-mini,gpt-5-petrobras,frozen_observed,simple,3,None,50,0.900,0.840,-0.060,FRACASSO,0.653
1,arc,phi4-mini,gpt-5-petrobras,frozen_observed,complex,3,None,50,0.900,0.760,-0.140,FRACASSO,0.653
2,gsm8k,phi4-mini,gpt-5-petrobras,frozen_observed,simple,3,None,50,0.860,0.800,-0.060,FRACASSO,0.687
3,gsm8k,phi4-mini,gpt-5-petrobras,frozen_observed,complex,3,None,50,0.860,0.780,-0.080,FRACASSO,0.687
4,arc,llama3-8b,gpt-5-petrobras,frozen_observed,simple,3,None,50,0.840,0.700,-0.140,FRACASSO,0.653
5,arc,llama3-8b,gpt-5-petrobras,frozen_observed,complex,3,None,50,0.840,0.720,-0.120,FRACASSO,0.653
6,gsm8k,llama3-8b,gpt-5-petrobras,frozen_observed,simple,3,None,50,0.820,0.760,-0.060,FRACASSO,0.687
7,gsm8k,llama3-8b,gpt-5-petrobras,frozen_observed,complex,3,None,50,0.820,0.540,-0.280,FRACASSO,0.687
8,arc,mistral-7b,gpt-5-petrobras,frozen_observed,simple,3,None,50,0.780,0.240,-0.540,FRACASSO,0.653
9,arc,mistral-7b,gpt-5-petrobras,frozen_observed,complex,3,None,50,0.780,0.340,-0.440,FRACASSO,0.653


## 3. Similaridade, k e prompt engineering

A celula seguinte implementa a grade real. Ela usa o cache `neighbors.npz` e as reflexoes GPT-5, limita a 30 itens e grava JSONL em `results/cache/debugger/`. Por padrao usa o backend `hf`, evitando o import de vLLM que pode falhar com algumas combinacoes de PyTorch; altere `RMCQ_DEBUGGER_BACKEND` para `vllm` somente se esse ambiente estiver validado. Por padrao nao executa inferencia: altere `RUN_GRID = True` conscientemente.

Os tres prompts sao: `frozen` (controle), `structured` (separa analise das alternativas) e `reflection_guard` (delimita reflexoes como orientacao, nao como resposta).

In [ ]:
from rmcq.backends import GenParams, get_backend
from rmcq.common import build_answer_prompt, build_eval_prompt, extract_final_answer, format_question
from rmcq.config import EMBEDDER, SEED, STUDENT_GEN
from rmcq.data import index_paths

BACKEND_KIND = os.environ.get('RMCQ_DEBUGGER_BACKEND', 'hf').lower()

# As variantes agora sao versoes do prompt central (rmcq.common), nao remendos
# em cima do prefixo. 'v1' e o layout antigo (licoes antes do enquadramento, sem
# enunciado de origem); 'v2' e o revisto para modelos pequenos: enquadramento
# primeiro, <notes> delimitadas com o enunciado de origem, questao e formato de
# saida por ultimo, reflexao cortada por orcamento e letras de alternativa
# neutralizadas. Ver rmcq/common.py, secao 1.
PROMPT_VARIANTS = ('v1', 'v2')

def find_index(dataset):
    candidates = sorted((ROOT / 'results' / 'index').glob(f'*/{dataset}/neighbors.npz'))
    if not candidates:
        raise FileNotFoundError(f'Indice ausente para {dataset}')
    return candidates[0]

def load_neighbors(dataset, required_k):
    """Carrega o cache ou amplia o ranking com os embeddings ja salvos."""
    neighbors_path = find_index(dataset)
    cached = np.load(neighbors_path, allow_pickle=False)
    train_uids = cached['train_uids'].tolist()
    test_uids = cached['test_uids'].tolist()
    if cached['top_idx'].shape[1] >= required_k:
        return train_uids, test_uids, cached['top_idx'], cached['top_sim']

    train_embeddings = np.load(neighbors_path.with_name('train.npy'))
    test_embeddings = np.load(neighbors_path.with_name('test.npy'))
    similarities = test_embeddings @ train_embeddings.T
    ranking = np.argsort(-similarities, axis=1, kind='stable')[:, :required_k]
    row_indices = np.arange(similarities.shape[0])[:, None]
    return train_uids, test_uids, ranking.astype(np.int32), similarities[row_indices, ranking].astype(np.float32)

def run_grid(students=STUDENTS, datasets=DATASETS, depths=DEPTHS, ks=K_VALUES, thresholds=THRESHOLDS, variants=PROMPT_VARIANTS, n=N_ITEMS, backend_kind=BACKEND_KIND):
    params = GenParams.from_config(STUDENT_GEN, seed=SEED)
    output = []
    for dataset in datasets:
        items = load_map(ROOT / 'data' / 'splits' / dataset / 'test.jsonl')
        selected = list(items.values())[:n]
        train = load_map(ROOT / 'data' / 'splits' / dataset / 'train.jsonl')
        max_k = max(ks)
        train_uids, test_uids, top_idx, top_sim = load_neighbors(dataset, max_k)
        pos = {uid: i for i, uid in enumerate(test_uids)}
        for student in students:
            base = load_map(ROOT / 'results' / 'baseline' / student / f'{dataset}_test.jsonl')
            with get_backend(student, kind=backend_kind) as backend:
                for depth in depths:
                    refl = load_map(ROOT / 'results' / 'reflections' / f'{student}__{TEACHER}__{depth}' / f'{dataset}.jsonl')
                    for k in ks:
                        for threshold in thresholds:
                            for variant in variants:
                                prompts, meta = [], []
                                for item in selected:
                                    row = pos[item['uid']]
                                    pairs = [(train_uids[j], float(s)) for j, s in zip(top_idx[row, :k], top_sim[row, :k]) if threshold is None or float(s) >= threshold]
                                    pairs = [(u, s) for u, s in pairs if (refl.get(u) or {}).get('reflection_text')]
                                    pairs.sort(key=lambda pair: pair[1])
                                    texts = [refl[u]['reflection_text'] for u, _ in pairs]
                                    # format_question: com o contexto, que e sobre o que a
                                    # similaridade do indice foi calculada. Antes daqui nao
                                    # ia enunciado de origem nenhum, e a nota chegava ao
                                    # aluno como conselho sem referente.
                                    source_questions = [format_question(train[u]) for u, _ in pairs]
                                    source_outcomes = [(refl[u].get('extra') or {}).get('source_was_correct') for u, _ in pairs]
                                    prompts.append(build_eval_prompt(item, texts, source_questions, source_outcomes, version=variant))
                                    meta.append((item, pairs))
                                generations = backend.generate(prompts, params, desc=f'debugger {student}/{dataset}/{depth}/k{k}/{variant}')
                                for (item, pairs), generation, prompt in zip(meta, generations, prompts):
                                    extraction = extract_final_answer(generation.text, item['choices'])
                                    output.append({
                                        'uid': item['uid'], 'dataset': dataset, 'student': student, 'teacher': TEACHER, 'depth': depth,
                                        'prompt_variant': variant, 'k': k, 'similarity_threshold': threshold,
                                        'predicted': extraction.letter, 'gold': item['answerKey'],
                                        'is_correct': extraction.letter == item['answerKey'],
                                        'n_reflections': len(pairs), 'top1_similarity': max((s for _, s in pairs), default=None),
                                        'prompt': prompt, 'raw_output': generation.text,
                                    })
    result = pd.DataFrame(output)
    out = ROOT / 'results' / 'cache' / 'debugger'
    out.mkdir(parents=True, exist_ok=True)
    result.to_json(out / 'responses.jsonl', orient='records', lines=True, force_ascii=False)
    return result

def summarize_grid(result, observed_baseline=None):
    base = {}
    for student in STUDENTS:
        for dataset in DATASETS:
            rows = load_map(ROOT / 'results' / 'baseline' / student / f'{dataset}_test.jsonl')
            selected = {item['uid'] for item in first_items(dataset)}
            base[(student, dataset)] = acc([rows[u] for u in selected if u in rows])
    grouped = result.groupby(['dataset', 'student', 'teacher', 'depth', 'prompt_variant', 'k', 'similarity_threshold'], dropna=False)
    summary = grouped['is_correct'].agg(n='size', reflection_accuracy='mean').reset_index()
    summary['baseline_accuracy'] = [base[(r.student, r.dataset)] for r in summary.itertuples()]
    summary['delta'] = summary['reflection_accuracy'] - summary['baseline_accuracy']
    summary['status'] = np.where(summary['delta'] > 0, 'SUCESSO', 'FRACASSO')
    return summary.sort_values(['dataset', 'student', 'delta'], ascending=[True, True, False])

RUN_GRID = True
if RUN_GRID:
    grid = run_grid()
    display(summarize_grid(grid))
else:
    print('Inferencia desativada. Defina RUN_GRID=True para testar a grade; resultados observados ja estao na tabela acima.')

## 4. Diagnostico de viradas

Depois de executar a grade, esta celula mostra quais configuracoes corrigem erros e quais transformam acertos em erros. Isso ajuda a distinguir reflexao util de uma mudanca aleatoria de resposta.

In [ ]:
def flip_report(result):
    rows = []
    for key, group in result.groupby(['dataset', 'student', 'depth', 'prompt_variant', 'k', 'similarity_threshold'], dropna=False):
        base = load_map(ROOT / 'results' / 'baseline' / key[1] / f'{key[0]}_test.jsonl')
        shared = group[group.uid.isin(base)]
        wrong_to_right = sum((not bool(base[r.uid].get('is_correct'))) and bool(r.is_correct) for r in shared.itertuples())
        right_to_wrong = sum(bool(base[r.uid].get('is_correct')) and (not bool(r.is_correct)) for r in shared.itertuples())
        rows.append(dict(zip(['dataset', 'student', 'depth', 'prompt_variant', 'k', 'similarity_threshold'], key), n=len(shared), wrong_to_right=wrong_to_right, right_to_wrong=right_to_wrong, utility=(wrong_to_right-right_to_wrong)/len(shared) if len(shared) else np.nan))
    return pd.DataFrame(rows).sort_values('utility', ascending=False)

if 'grid' in globals():
    display(flip_report(grid).head(20))
else:
    print('Execute a grade primeiro para obter o relatorio de viradas.')